# 🚀 Production Deployment

**Deploy fine-tuned models to production**

---

## 📋 Overview

**What you'll learn:**
- Deployment strategies
- API endpoints setup
- Load balancing
- Monitoring and alerting
- Rollback procedures

**Time estimate:** ⏱️ 45 minutes | **Difficulty:** 🔴 Advanced

---

## 🤔 Deployment Strategies

### Deployment Options:

**1. OpenAI API (Easiest)**
```python
✅ Pros:
- Zero infrastructure
- Auto-scaling
- High availability

❌ Cons:
- Ongoing API costs
- Internet required
- Less control
```

**2. Self-Hosted (Most Control)**
```python
✅ Pros:
- Full control
- No per-request costs
- Private/offline

❌ Cons:
- Infrastructure overhead
- Scaling complexity
- Maintenance burden
```

**3. Hybrid (Best of Both)**
```python
✅ Use cases:
- OpenAI for spikes
- Self-hosted for baseline
- Failover capability
```

## 🎯 OpenAI API Deployment

In [ ]:
from openai import OpenAI
import os
import time
from typing import Dict, List
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class ProductionLLMAPI:
    """Production-ready LLM API wrapper."""
    
    def __init__(
        self,
        model_id: str,
        fallback_model: str = "gpt-3.5-turbo"
    ):
        self.model_id = model_id
        self.fallback_model = fallback_model
        self.client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
        
        # Metrics
        self.request_count = 0
        self.error_count = 0
        self.fallback_count = 0
    
    def generate(
        self,
        messages: List[Dict],
        temperature: float = 0.7,
        max_retries: int = 3
    ) -> Dict:
        """Generate response with retries and fallback."""
        
        self.request_count += 1
        
        for attempt in range(max_retries):
            try:
                start_time = time.time()
                
                response = self.client.chat.completions.create(
                    model=self.model_id,
                    messages=messages,
                    temperature=temperature
                )
                
                latency = time.time() - start_time
                
                logger.info(f"Success: {latency:.2f}s")
                
                return {
                    'success': True,
                    'content': response.choices[0].message.content,
                    'model': self.model_id,
                    'latency': latency,
                    'usage': {
                        'prompt_tokens': response.usage.prompt_tokens,
                        'completion_tokens': response.usage.completion_tokens,
                    }
                }
            
            except Exception as e:
                logger.error(f"Attempt {attempt + 1} failed: {e}")
                
                if attempt == max_retries - 1:
                    # Final attempt - try fallback
                    logger.warning(f"Falling back to {self.fallback_model}")
                    return self._fallback_generate(messages, temperature)
                
                # Wait before retry (exponential backoff)
                time.sleep(2 ** attempt)
        
        self.error_count += 1
        return {
            'success': False,
            'error': 'Max retries exceeded'
        }
    
    def _fallback_generate(self, messages: List[Dict], temperature: float) -> Dict:
        """Fallback to base model."""
        
        self.fallback_count += 1
        
        try:
            response = self.client.chat.completions.create(
                model=self.fallback_model,
                messages=messages,
                temperature=temperature
            )
            
            return {
                'success': True,
                'content': response.choices[0].message.content,
                'model': self.fallback_model,
                'fallback': True
            }
        except Exception as e:
            self.error_count += 1
            return {
                'success': False,
                'error': str(e)
            }
    
    def get_metrics(self) -> Dict:
        """Get API metrics."""
        return {
            'requests': self.request_count,
            'errors': self.error_count,
            'fallbacks': self.fallback_count,
            'error_rate': self.error_count / max(self.request_count, 1) * 100,
            'fallback_rate': self.fallback_count / max(self.request_count, 1) * 100,
        }

print("🚀 Production API Example\n")
print("""
# Initialize
api = ProductionLLMAPI(
    model_id="ft:gpt-3.5-turbo:company:v1:abc",
    fallback_model="gpt-3.5-turbo"
)

# Make request
result = api.generate(
    messages=[{"role": "user", "content": "Hello"}]
)

if result['success']:
    print(result['content'])
    print(f"Latency: {result['latency']:.2f}s")

# Check metrics
metrics = api.get_metrics()
print(metrics)
""")

## 🔄 Canary Deployment

In [ ]:
import random

class CanaryDeployment:
    """Gradual rollout with canary deployment."""
    
    def __init__(
        self,
        stable_model: str,
        canary_model: str,
        canary_percentage: float = 10.0
    ):
        self.stable_model = stable_model
        self.canary_model = canary_model
        self.canary_percentage = canary_percentage
        
        self.stable_api = ProductionLLMAPI(stable_model)
        self.canary_api = ProductionLLMAPI(canary_model)
    
    def route_request(self, messages: List[Dict]) -> Dict:
        """Route to canary or stable based on percentage."""
        
        if random.random() * 100 < self.canary_percentage:
            # Route to canary
            logger.info("Routing to CANARY")
            result = self.canary_api.generate(messages)
            result['deployment'] = 'canary'
            return result
        else:
            # Route to stable
            logger.info("Routing to STABLE")
            result = self.stable_api.generate(messages)
            result['deployment'] = 'stable'
            return result
    
    def increase_canary(self, percentage: float):
        """Gradually increase canary traffic."""
        self.canary_percentage = min(100.0, percentage)
        logger.info(f"Canary traffic: {self.canary_percentage}%")
    
    def rollback(self):
        """Rollback to stable (0% canary)."""
        self.canary_percentage = 0.0
        logger.warning("ROLLBACK: 100% stable traffic")
    
    def promote_canary(self):
        """Promote canary to stable."""
        self.stable_model = self.canary_model
        self.canary_percentage = 0.0
        logger.info("PROMOTED: Canary → Stable")
    
    def get_metrics(self) -> Dict:
        """Compare canary vs stable."""
        return {
            'canary_percentage': self.canary_percentage,
            'stable': self.stable_api.get_metrics(),
            'canary': self.canary_api.get_metrics(),
        }

print("🔄 Canary Deployment Example\n")
print("""
# Rollout plan
deployment = CanaryDeployment(
    stable_model="ft:gpt-3.5-turbo:company:v1:abc",
    canary_model="ft:gpt-3.5-turbo:company:v2:def",
    canary_percentage=5.0  # Start with 5%
)

# Day 1: 5% canary
for _ in range(100):
    deployment.route_request(messages)
# Monitor metrics

# Day 2: Increase to 25% if metrics good
if canary_metrics_good():
    deployment.increase_canary(25.0)

# Day 3: 50%
deployment.increase_canary(50.0)

# Day 4: 100% (full rollout)
deployment.increase_canary(100.0)

# Day 5: Promote canary to stable
deployment.promote_canary()

# If issues detected at any point:
deployment.rollback()  # Back to 100% stable
""")

## 📊 Monitoring & Alerting

In [ ]:
from collections import deque
from dataclasses import dataclass
from datetime import datetime

@dataclass
class Alert:
    severity: str  # 'warning' or 'critical'
    message: str
    timestamp: datetime

class ProductionMonitor:
    """Monitor production LLM performance."""
    
    def __init__(self, window_size: int = 100):
        self.latencies = deque(maxlen=window_size)
        self.errors = deque(maxlen=window_size)
        self.window_size = window_size
        self.alerts = []
    
    def record(self, success: bool, latency: float = None):
        """Record request result."""
        if latency:
            self.latencies.append(latency)
        self.errors.append(0 if success else 1)
    
    def check_health(self) -> Dict:
        """Check system health."""
        if not self.errors:
            return {'healthy': True}
        
        import numpy as np
        
        # Calculate metrics
        error_rate = sum(self.errors) / len(self.errors) * 100
        
        if self.latencies:
            p50_latency = np.percentile(list(self.latencies), 50)
            p99_latency = np.percentile(list(self.latencies), 99)
        else:
            p50_latency = 0
            p99_latency = 0
        
        # Check thresholds
        alerts = []
        
        if error_rate > 5.0:
            alert = Alert(
                severity='critical',
                message=f"High error rate: {error_rate:.1f}%",
                timestamp=datetime.now()
            )
            alerts.append(alert)
            self.alerts.append(alert)
        
        if p99_latency > 5.0:  # 5 seconds
            alert = Alert(
                severity='warning',
                message=f"High P99 latency: {p99_latency:.1f}s",
                timestamp=datetime.now()
            )
            alerts.append(alert)
            self.alerts.append(alert)
        
        return {
            'healthy': len(alerts) == 0,
            'error_rate': error_rate,
            'p50_latency': p50_latency,
            'p99_latency': p99_latency,
            'alerts': [{
                'severity': a.severity,
                'message': a.message
            } for a in alerts]
        }

print("📊 Production Monitoring Example\n")
print("""
monitor = ProductionMonitor()

# Record requests
for request in incoming_requests:
    start = time.time()
    result = api.generate(request)
    latency = time.time() - start
    
    monitor.record(
        success=result['success'],
        latency=latency
    )

# Check health every minute
health = monitor.check_health()

if not health['healthy']:
    for alert in health['alerts']:
        if alert['severity'] == 'critical':
            send_pager_duty(alert)
        else:
            send_slack(alert)

Example output:
{
  'healthy': False,
  'error_rate': 8.5%,
  'p50_latency': 0.8s,
  'p99_latency': 3.2s,
  'alerts': [
    {'severity': 'critical', 'message': 'High error rate: 8.5%'}
  ]
}
""")

## ✅ Summary

### Production Deployment Checklist:

**Pre-Deployment:**
```python
✅ Model tested on validation set
✅ A/B test shows improvement
✅ Monitoring configured
✅ Alerts set up
✅ Rollback plan ready
✅ Load testing complete
```

**Deployment:**
```python
Day 1: 5% canary traffic
Day 2: 25% if metrics good
Day 3: 50%
Day 4: 100%
Day 5: Promote to stable
```

**Post-Deployment:**
```python
✅ Monitor error rates
✅ Track latency (P50, P99)
✅ Watch user feedback
✅ Compare to baseline
✅ Document learnings
```

### Key Metrics:

| Metric | Target | Alert |
|--------|--------|-------|
| **Error rate** | < 1% | > 5% |
| **P50 latency** | < 1s | > 3s |
| **P99 latency** | < 3s | > 5s |
| **Availability** | > 99.9% | < 99% |

### Rollback Triggers:

```python
# Automatic rollback if:
- Error rate > 10%
- P99 latency > 10s
- User complaints spike
- Model produces unsafe content
```

### Best Practices:

**1. Gradual Rollout**
- Never 0% → 100%
- Start with 5-10%
- Double each day if healthy

**2. Fallback Model**
- Always have a backup
- Test fallback regularly
- Monitor fallback usage

**3. Comprehensive Monitoring**
- Technical metrics (latency, errors)
- Business metrics (satisfaction, completion)
- Quality metrics (accuracy, helpfulness)

**4. Quick Rollback**
- One-click rollback
- Test rollback procedure
- Document rollback process

### Congratulations! 🎉

You've completed the Fine-Tuning module!

**You learned:**
- When to fine-tune vs RAG
- Data preparation & quality
- OpenAI & LoRA fine-tuning
- Evaluation & metrics
- Hyperparameter tuning
- Synthetic data generation
- Production deployment

**Next module:** `07_agents_tools/` - Building AI agents with tools